# #166 Research Smoke Test — Cache-first SDU/yfinance/DuckDB/FinRL proof

This notebook demonstrates the research side of #166.

It is deliberately **cache-first**:

1. Read already-ingested system data from `system/runtime-data/market_research.duckdb`.
2. If the cache is incomplete, try `SDU_DataScienceTool`.
3. If SDU/Yahoo fails, try direct `yfinance`.
4. Write CSV.
5. Write Parquet.
6. Write a research DuckDB table.
7. Reload from DuckDB.
8. Produce a FinRL-compatible dataframe.

This avoids failing the research proof when Yahoo/yfinance is rate-limited or has SSL/network problems.


## Setup

Run from repo root:

```powershell
conda activate stockdss
pip install -r research/requirements-market-data.txt
jupyter notebook research/notebooks/166_finrl_yfinance_sdu_duckdb_cache_first_smoke.ipynb
```

If `pip install` complains about the package name, make sure the requirement line is:

```text
sdu-dst @ git+https://github.com/guldmand/SDU_DataScienceTool.git
```


In [2]:
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 180)

def find_repo_root(start: Path | None = None) -> Path:
    """
    Find StockInvestmentDSS repo root by walking upward until we find
    the expected project folders.
    """
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "system").exists()
            and (candidate / "research").exists()
            and (candidate / "project-management").exists()
        ):
            return candidate

    raise RuntimeError(
        f"Could not find repo root from {current}. "
        "Open VS Code from the StockInvestmentDSS root or adjust find_repo_root()."
    )

REPO_ROOT = find_repo_root()

DUCKDB_PATH = REPO_ROOT / "system" / "runtime-data" / "market_research.duckdb"
ARTIFACT_DIR = REPO_ROOT / "research" / "experiments" / "artifacts" / "market_data_smoke"

TICKERS = ["AAPL", "MSFT", "SPY", "NVDA"]
START_DATE = "2025-01-01"
END_DATE = "2026-01-01"

RUN_LIVE_FETCH_IF_CACHE_COMPLETE = False

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DUCKDB_PATH.parent.mkdir(parents=True, exist_ok=True)

print("cwd:         ", Path.cwd())
print("repo root:   ", REPO_ROOT)
print("duckdb path: ", DUCKDB_PATH)
print("artifact dir:", ARTIFACT_DIR)
print("duckdb exists:", DUCKDB_PATH.exists())

cwd:          c:\Users\gurug\Dropbox\DataScience\Speciale\Devices\dev\StockInvestmentDSS\research\notebooks
repo root:    C:\Users\gurug\Dropbox\DataScience\Speciale\Devices\dev\StockInvestmentDSS
duckdb path:  C:\Users\gurug\Dropbox\DataScience\Speciale\Devices\dev\StockInvestmentDSS\system\runtime-data\market_research.duckdb
artifact dir: C:\Users\gurug\Dropbox\DataScience\Speciale\Devices\dev\StockInvestmentDSS\research\experiments\artifacts\market_data_smoke
duckdb exists: True


## Helper functions

In [4]:
OHLCV_COLUMNS = ["open", "high", "low", "close", "volume"]

import datetime
from datetime import datetime, timezone
from typing import Any


def flatten_column_name(column: Any) -> str:
    if isinstance(column, tuple):
        parts = [
            str(part).strip().lower().replace(" ", "_")
            for part in column
            if str(part).strip() and str(part).strip().lower() != "none"
        ]
        return "_".join(parts)
    return str(column).strip().lower().replace(" ", "_")


def find_column(columns: list[str], ticker: str, canonical_name: str, aliases: list[str] | None = None) -> str | None:
    aliases = aliases or []
    ticker_lower = ticker.lower()
    candidates = [
        canonical_name,
        f"{canonical_name}_{ticker_lower}",
        f"{ticker_lower}_{canonical_name}",
        *aliases,
        *[f"{alias}_{ticker_lower}" for alias in aliases],
        *[f"{ticker_lower}_{alias}" for alias in aliases],
    ]
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


def normalize_one_ticker_frame(frame: pd.DataFrame, ticker: str, source: str) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()

    ticker_upper = ticker.strip().upper()
    normalized = frame.copy()

    if normalized.index.name is not None:
        normalized = normalized.reset_index()

    normalized.columns = [flatten_column_name(column) for column in normalized.columns]
    normalized = normalized.loc[:, ~normalized.columns.duplicated()]
    columns = list(normalized.columns)

    date_column = next((c for c in ["date", "datetime", "timestamp", "price_date", "index"] if c in columns), None)
    if date_column is None:
        raise ValueError(f"Missing date column for {ticker_upper}. Available columns: {columns}")

    column_map = {
        "open": find_column(columns, ticker_upper, "open"),
        "high": find_column(columns, ticker_upper, "high"),
        "low": find_column(columns, ticker_upper, "low"),
        "close": find_column(columns, ticker_upper, "close"),
        "volume": find_column(columns, ticker_upper, "volume"),
        "adj_close": find_column(columns, ticker_upper, "adj_close", aliases=["adjusted_close", "adjclose"]),
    }

    missing = [col for col in OHLCV_COLUMNS if column_map[col] is None]
    if missing:
        raise ValueError(f"Missing OHLCV columns for {ticker_upper}: {missing}. Available columns: {columns}")

    if column_map["adj_close"] is None:
        column_map["adj_close"] = column_map["close"]

    out = pd.DataFrame({
        "date": pd.to_datetime(normalized[date_column], errors="coerce").dt.date,
        "ticker": ticker_upper,
        "open": pd.to_numeric(normalized[column_map["open"]], errors="coerce"),
        "high": pd.to_numeric(normalized[column_map["high"]], errors="coerce"),
        "low": pd.to_numeric(normalized[column_map["low"]], errors="coerce"),
        "close": pd.to_numeric(normalized[column_map["close"]], errors="coerce"),
        "adj_close": pd.to_numeric(normalized[column_map["adj_close"]], errors="coerce"),
        "volume": pd.to_numeric(normalized[column_map["volume"]], errors="coerce"),
        "source": source,
        "ingested_at": datetime.now(timezone.utc).isoformat(),
    })

    out = out.dropna(subset=["date", "open", "high", "low", "close", "volume"])
    out["ticker"] = out["ticker"].astype(str).str.upper()
    out["volume"] = out["volume"].fillna(0).astype("int64")

    return out.sort_values(["ticker", "date"]).reset_index(drop=True)


def to_finrl_price_frame(prices: pd.DataFrame) -> pd.DataFrame:
    if prices.empty:
        return pd.DataFrame(columns=["date", "open", "high", "low", "close", "volume", "tic", "day"])

    frame = prices.copy()
    frame["date"] = pd.to_datetime(frame["date"])
    frame["tic"] = frame["ticker"].astype(str).str.upper()
    frame["day"] = frame["date"].dt.dayofweek
    frame["date"] = frame["date"].dt.strftime("%Y-%m-%d")
    return frame[["date", "open", "high", "low", "close", "volume", "tic", "day"]].sort_values(["date", "tic"]).reset_index(drop=True)


## 1. Read system-track DuckDB cache first

This uses the market data you already ingested through the backend endpoints. It avoids repeated Yahoo calls during notebook testing.


In [5]:
import duckdb

def read_system_cache(duckdb_path: Path, tickers: list[str]) -> pd.DataFrame:
    if not duckdb_path.exists():
        return pd.DataFrame()

    ticker_list = [ticker.upper() for ticker in tickers]

    with duckdb.connect(str(duckdb_path)) as connection:
        table_count = connection.execute(
            """
            SELECT COUNT(*)
            FROM information_schema.tables
            WHERE table_name = 'market_prices_daily'
            """
        ).fetchone()[0]

        if table_count == 0:
            return pd.DataFrame()

        frame = connection.execute(
            """
            SELECT
                ticker,
                price_date AS date,
                open,
                high,
                low,
                close,
                adj_close,
                volume,
                source,
                ingested_at
            FROM market_prices_daily
            WHERE ticker IN (SELECT * FROM UNNEST(?))
              AND price_date >= ?
              AND price_date < ?
            ORDER BY price_date, ticker
            """,
            [ticker_list, START_DATE, END_DATE],
        ).df()

    if frame.empty:
        return frame

    frame["ticker"] = frame["ticker"].astype(str).str.upper()
    return frame


cache_prices = read_system_cache(DUCKDB_PATH, TICKERS)
cache_tickers = sorted(cache_prices["ticker"].unique().tolist()) if not cache_prices.empty else []

print("cache rows:", len(cache_prices))
print("cache tickers:", cache_tickers)

display(cache_prices.head())


cache rows: 1000
cache tickers: ['AAPL', 'MSFT', 'NVDA', 'SPY']


,ticker,date,open,high,low,close,adj_close,volume,source,ingested_at
0,AAPL,2025-01-02,248.929993,249.100006,241.820007,243.850006,242.301941,55740700,yfinance:fallback-after-sdu-error,2026-05-12 22:18:54.206933
1,MSFT,2025-01-02,425.529999,426.070007,414.850006,418.579987,414.568634,16896500,yfinance:fallback-after-sdu-error,2026-05-12 22:23:57.691465
2,NVDA,2025-01-02,136.000000,138.880005,134.630005,138.309998,138.264694,198247200,yfinance:fallback-after-sdu-error,2026-05-12 22:24:56.579134
3,SPY,2025-01-02,589.390015,591.130005,580.500000,584.640015,576.280334,50204000,yfinance:fallback-after-sdu-error,2026-05-12 22:24:12.085054
4,AAPL,2025-01-03,243.360001,244.179993,241.889999,243.360001,241.815033,40244100,yfinance:fallback-after-sdu-error,2026-05-12 22:18:54.206933


## 2. Only fetch live data if cache is incomplete

Yahoo/yfinance can rate-limit or throw transient SSL errors. This cell therefore only calls external data when the local DuckDB cache is missing required tickers, unless `RUN_LIVE_FETCH_IF_CACHE_COMPLETE = True`.


In [6]:
import time

async def fetch_with_sdu_datascience_tool(tickers: list[str], start_date: str, end_date: str) -> tuple[pd.DataFrame, str, str | None]:
    try:
        from sdu_dst.sources.yahoo import YahooFinanceSource

        source = YahooFinanceSource()
        frames = []

        for ticker in tickers:
            raw = await source.fetch_prices([ticker], start_date, end_date)
            frames.append(normalize_one_ticker_frame(raw, ticker, "sdu-dst:yahoo"))
            time.sleep(1.0)

        prices = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        if prices.empty:
            return prices, "sdu-dst:yahoo", "SDU_DataScienceTool returned no rows."

        return prices, "sdu-dst:yahoo", None

    except Exception as exc:
        return pd.DataFrame(), "sdu-dst:yahoo", str(exc)


async def fetch_with_yfinance_direct(tickers: list[str], start_date: str, end_date: str) -> tuple[pd.DataFrame, str, str | None]:
    try:
        import yfinance as yf

        frames = []
        for ticker in tickers:
            raw = yf.download(
                ticker,
                start=start_date,
                end=end_date,
                interval="1d",
                auto_adjust=False,
                progress=False,
                group_by="column",
                threads=False,
            )
            if raw is None or raw.empty:
                continue

            raw = raw.reset_index()
            raw["ticker"] = ticker.upper()
            frames.append(normalize_one_ticker_frame(raw, ticker, "yfinance:research-fallback"))
            time.sleep(1.0)

        prices = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        if prices.empty:
            return prices, "yfinance:research-fallback", "yfinance returned no rows."

        return prices, "yfinance:research-fallback", None

    except Exception as exc:
        return pd.DataFrame(), "yfinance:research-fallback", str(exc)


required = {ticker.upper() for ticker in TICKERS}
available = set(cache_tickers)
missing = sorted(required.difference(available))

if cache_prices.empty or missing or RUN_LIVE_FETCH_IF_CACHE_COMPLETE:
    fetch_tickers = missing if missing else TICKERS
    print("Live fetch needed for:", fetch_tickers)

    fetched, source_used, message = await fetch_with_sdu_datascience_tool(fetch_tickers, START_DATE, END_DATE)

    if fetched.empty:
        print("SDU fetch failed or empty:", message)
        print("Trying direct yfinance fallback...")
        fetched, source_used, message = await fetch_with_yfinance_direct(fetch_tickers, START_DATE, END_DATE)

    if fetched.empty:
        print("External fetch failed or empty:", message)
        print("Using only existing DuckDB cache.")
        prices = cache_prices
        source_used = "duckdb:existing-system-cache"
    else:
        prices = pd.concat([cache_prices, fetched], ignore_index=True) if not cache_prices.empty else fetched
        prices = prices.drop_duplicates(subset=["ticker", "date"]).sort_values(["date", "ticker"]).reset_index(drop=True)
else:
    prices = cache_prices
    source_used = "duckdb:existing-system-cache"

if prices.empty:
    raise RuntimeError("No data available from system DuckDB cache, SDU_DataScienceTool, or yfinance.")

print("source used:", source_used)
print("rows:", len(prices))
print("tickers:", sorted(prices["ticker"].unique().tolist()))
display(prices.head())


source used: duckdb:existing-system-cache
rows: 1000
tickers: ['AAPL', 'MSFT', 'NVDA', 'SPY']


,ticker,date,open,high,low,close,adj_close,volume,source,ingested_at
0,AAPL,2025-01-02,248.929993,249.100006,241.820007,243.850006,242.301941,55740700,yfinance:fallback-after-sdu-error,2026-05-12 22:18:54.206933
1,MSFT,2025-01-02,425.529999,426.070007,414.850006,418.579987,414.568634,16896500,yfinance:fallback-after-sdu-error,2026-05-12 22:23:57.691465
2,NVDA,2025-01-02,136.000000,138.880005,134.630005,138.309998,138.264694,198247200,yfinance:fallback-after-sdu-error,2026-05-12 22:24:56.579134
3,SPY,2025-01-02,589.390015,591.130005,580.500000,584.640015,576.280334,50204000,yfinance:fallback-after-sdu-error,2026-05-12 22:24:12.085054
4,AAPL,2025-01-03,243.360001,244.179993,241.889999,243.360001,241.815033,40244100,yfinance:fallback-after-sdu-error,2026-05-12 22:18:54.206933


## 3. Write CSV and Parquet

In [7]:
csv_path = ARTIFACT_DIR / "market_prices_daily.csv"
parquet_path = ARTIFACT_DIR / "market_prices_daily.parquet"

prices.to_csv(csv_path, index=False)
prices.to_parquet(parquet_path, index=False)

print("CSV written:", csv_path)
print("Parquet written:", parquet_path)


CSV written: C:\Users\gurug\Dropbox\DataScience\Speciale\Devices\dev\StockInvestmentDSS\research\experiments\artifacts\market_data_smoke\market_prices_daily.csv
Parquet written: C:\Users\gurug\Dropbox\DataScience\Speciale\Devices\dev\StockInvestmentDSS\research\experiments\artifacts\market_data_smoke\market_prices_daily.parquet


## 4. Write research table to DuckDB

In [8]:
def write_research_prices(prices: pd.DataFrame, duckdb_path: Path) -> int:
    frame = prices.copy()
    frame["price_date"] = pd.to_datetime(frame["date"]).dt.date

    insert_frame = frame[
        [
            "ticker",
            "price_date",
            "open",
            "high",
            "low",
            "close",
            "adj_close",
            "volume",
            "source",
            "ingested_at",
        ]
    ]

    with duckdb.connect(str(duckdb_path)) as connection:
        connection.execute(
            """
            CREATE TABLE IF NOT EXISTS research_market_prices_daily (
                ticker VARCHAR NOT NULL,
                price_date DATE NOT NULL,
                open DOUBLE NOT NULL,
                high DOUBLE NOT NULL,
                low DOUBLE NOT NULL,
                close DOUBLE NOT NULL,
                adj_close DOUBLE,
                volume BIGINT NOT NULL,
                source VARCHAR NOT NULL,
                ingested_at TIMESTAMP NOT NULL,
                PRIMARY KEY (ticker, price_date)
            );
            """
        )

        connection.register("research_prices_df", insert_frame)

        connection.execute(
            """
            DELETE FROM research_market_prices_daily
            WHERE ticker IN (SELECT DISTINCT ticker FROM research_prices_df)
            """
        )

        connection.execute(
            """
            INSERT INTO research_market_prices_daily
            SELECT * FROM research_prices_df
            """
        )

    return int(len(insert_frame))


rows_written = write_research_prices(prices, DUCKDB_PATH)
print("DuckDB research rows written:", rows_written)


DuckDB research rows written: 1000


## 5. Reload from DuckDB

In [9]:
with duckdb.connect(str(DUCKDB_PATH)) as connection:
    reloaded = connection.execute(
        """
        SELECT
            ticker,
            price_date AS date,
            open,
            high,
            low,
            close,
            adj_close,
            volume,
            source,
            ingested_at
        FROM research_market_prices_daily
        WHERE ticker IN (SELECT * FROM UNNEST(?))
        ORDER BY price_date, ticker
        """,
        [TICKERS],
    ).df()

print("rows reloaded:", len(reloaded))
display(reloaded.head())


rows reloaded: 1000


,ticker,date,open,high,low,close,adj_close,volume,source,ingested_at
0,AAPL,2025-01-02,248.929993,249.100006,241.820007,243.850006,242.301941,55740700,yfinance:fallback-after-sdu-error,2026-05-12 22:18:54.206933
1,MSFT,2025-01-02,425.529999,426.070007,414.850006,418.579987,414.568634,16896500,yfinance:fallback-after-sdu-error,2026-05-12 22:23:57.691465
2,NVDA,2025-01-02,136.000000,138.880005,134.630005,138.309998,138.264694,198247200,yfinance:fallback-after-sdu-error,2026-05-12 22:24:56.579134
3,SPY,2025-01-02,589.390015,591.130005,580.500000,584.640015,576.280334,50204000,yfinance:fallback-after-sdu-error,2026-05-12 22:24:12.085054
4,AAPL,2025-01-03,243.360001,244.179993,241.889999,243.360001,241.815033,40244100,yfinance:fallback-after-sdu-error,2026-05-12 22:18:54.206933


## 6. Build FinRL-compatible dataframe

In [10]:
finrl_prices = to_finrl_price_frame(reloaded)

finrl_csv_path = ARTIFACT_DIR / "finrl_prices.csv"
finrl_prices.to_csv(finrl_csv_path, index=False)

required_columns = {"date", "open", "high", "low", "close", "volume", "tic", "day"}
missing = required_columns.difference(finrl_prices.columns)

assert not missing, f"Missing FinRL columns: {missing}"
assert not finrl_prices.empty, "FinRL dataframe is empty"

print("FinRL CSV written:", finrl_csv_path)
print("rows:", len(finrl_prices))
print("columns:", list(finrl_prices.columns))
display(finrl_prices.head(12))


FinRL CSV written: C:\Users\gurug\Dropbox\DataScience\Speciale\Devices\dev\StockInvestmentDSS\research\experiments\artifacts\market_data_smoke\finrl_prices.csv
rows: 1000
columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'tic', 'day']


,date,open,high,low,close,volume,tic,day
0,2025-01-02,248.929993,249.100006,241.820007,243.850006,55740700,AAPL,3
1,2025-01-02,425.529999,426.070007,414.850006,418.579987,16896500,MSFT,3
2,2025-01-02,136.000000,138.880005,134.630005,138.309998,198247200,NVDA,3
3,2025-01-02,589.390015,591.130005,580.500000,584.640015,50204000,SPY,3
4,2025-01-03,243.360001,244.179993,241.889999,243.360001,40244100,AAPL,4
5,2025-01-03,421.079987,424.029999,419.540009,423.350006,16662900,MSFT,4
6,2025-01-03,140.009995,144.899994,139.729996,144.470001,229322500,NVDA,4
7,2025-01-03,587.530029,592.599976,586.429993,591.950012,37888500,SPY,4
8,2025-01-06,244.309998,247.330002,243.199997,245.000000,45045600,AAPL,0
9,2025-01-06,428.000000,434.320007,425.480011,427.850006,20573600,MSFT,0


## 7. Optional FinRL import check

This notebook does not start training. This check only reports whether the current environment has FinRL installed.


In [11]:
try:
    import finrl  # type: ignore
    print("FinRL import OK:", getattr(finrl, "__version__", "version unknown"))
except Exception as exc:
    print("FinRL not installed in this environment. OK for #166; dataframe is FinRL-compatible.")
    print("Import error:", exc)


FinRL import OK: version unknown


## Conclusion

If this notebook runs, #166 has a research-track proof:

```text
system DuckDB cache / SDU_DataScienceTool / yfinance
-> CSV
-> Parquet
-> DuckDB research table
-> reload
-> FinRL-compatible dataframe
```

No RL training is started.
